# 고급 Beta-VAE 파이프라인 (v2 - 성능 개선 전략 적용)

이 노트북은 이전 훈련에서 발견된 **'잠재 공간 비활성화(Latent Collapse)'** 문제를 해결하고, 최종 목표인 **검색 성능(Recall)을 극대화**하기 위한 고급 전략들을 적용한 개선된 버전입니다.

### 🌟 적용된 핵심 개선 전략 🌟

1.  **복합 손실 함수 (`MSE` + `Cosine Similarity`)**: 벡터의 값뿐만 아니라 **방향**까지 학습하여 의미적 구조를 보존하고 검색 성능을 직접적으로 향상시킵니다.
2.  **KL Annealing**: 훈련 초기에 KL Divergence의 가중치를 점진적으로 높여, 모델이 충분한 표현력을 학습한 후 잠재 공간을 안정적으로 형성하도록 유도합니다.
3.  **상세 모니터링**: 훈련 중 코사인 유사도 추이를 함께 추적하여 모델이 의미적으로 학습하고 있는지 다각도로 평가합니다.
4.  **잠재 공간 시각화 (`UMAP`)**: 훈련된 모델이 생성한 임베딩이 클래스(레이블)별로 잘 군집화되었는지 시각적으로 확인하여 임베딩의 품질을 정성적으로 평가합니다.
5.  **Recall@K 성능 측정 (`faiss`)**: 실제 검색 환경과 유사한 ANN(근사 근접 이웃) 검색을 시뮬레이션하여, 생성된 임베딩의 실용적인 검색 성능을 정량적으로 측정합니다.

In [1]:
# Step 0: 필요한 추가 라이브러리 설치
# !pip install umap-learn faiss-cpu matplotlib

In [2]:
# Import 라이브러리
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import psycopg2
import json
from datetime import datetime
import logging
from tqdm import tqdm
import os
import math
import copy
from sklearn.model_selection import train_test_split
import pandas as pd
import umap
import faiss
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("=" * 70)
print("🚀 Beta-VAE 고급 통합 임베딩 파이프라인 v2 시작")
print("🔥 복합 손실 함수 | KL Annealing | UMAP 시각화 | Recall@K 측정")
print("=" * 70)

# GPU 설정 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"디바이스 설정 완료: {device}")
print(f"🎯 사용 디바이스: {device}")

2025-06-23 06:24:22,377 - INFO - 디바이스 설정 완료: cuda


🚀 Beta-VAE 고급 통합 임베딩 파이프라인 v2 시작
🔥 복합 손실 함수 | KL Annealing | UMAP 시각화 | Recall@K 측정
🎯 사용 디바이스: cuda


## 1단계: 데이터 로딩 (레이블 포함)

UMAP 시각화 및 Recall 측정을 위해 `origin_vector` 테이블과 JOIN하여 **레이블 정보**를 함께 로드하도록 수정합니다.

In [3]:
# DB 연결
try:
    logger.info("PostgreSQL 데이터베이스 연결 시도 중...")
    conn = psycopg2.connect(
        dbname='postgres', user='postgres', password='postgres', host='localhost', port=5432
    )
    cursor = conn.cursor()
    logger.info("✅ 데이터베이스 연결 성공")
except Exception as e:
    logger.error(f"❌ 데이터베이스 연결 실패: {e}")
    raise

# 데이터 로드 (레이블 정보 포함)
def load_vectors_with_labels(table_name: str, dim_limit: int = 512):
    logger.info(f"테이블 '{table_name}'에서 벡터와 레이블 로딩 시작 (차원 제한: 128-{dim_limit})")
    query = f"""
        SELECT t.embedding, o.label
        FROM {table_name} t
        JOIN origin_vector o ON t.origin_vector_id = o.id
        WHERE t.embedding IS NOT NULL AND t.compressed_dim BETWEEN 128 AND {dim_limit}
          AND t.compression_ratio <= 1.0
    """
    cursor.execute(query)
    rows = cursor.fetchall()
    logger.info(f"쿼리 실행 완료: {len(rows)}개 레코드 발견")
    
    vectors, labels = [], []
    for row in tqdm(rows, desc=f"   {table_name} 파싱"):
        try:
            vectors.append(np.array(json.loads(row[0])))
            labels.append(row[1])
        except Exception as e:
            logger.warning(f"데이터 파싱 실패: {e}")
            
    logger.info(f"✅ '{table_name}' 로딩 완료: {len(vectors)}개 벡터")
    return vectors, labels

wavelet_vectors, wavelet_labels = load_vectors_with_labels('wavelet_vector')
dct_vectors, dct_labels = load_vectors_with_labels('dct_vector')

all_vectors = wavelet_vectors + dct_vectors
all_labels = wavelet_labels + dct_labels
original_dims = [len(vec) for vec in all_vectors]
max_dim = max(original_dims) if original_dims else 0

logger.info(f"데이터 로딩 완료: 총 {len(all_vectors)}개 벡터, 최대 차원 {max_dim}")
print(f"\n📈 데이터 요약:")
print(f"   🔸 총 벡터 수: {len(all_vectors):,}개")
print(f"   🔸 총 레이블 수: {len(all_labels):,}개")
print(f"   🔸 고유 레이블 수: {len(set(all_labels)):,}개")
print(f"   🔸 최대 차원: {max_dim}")

2025-06-23 06:24:22,398 - INFO - PostgreSQL 데이터베이스 연결 시도 중...
2025-06-23 06:24:22,416 - INFO - ✅ 데이터베이스 연결 성공
2025-06-23 06:24:22,416 - INFO - 테이블 'wavelet_vector'에서 벡터와 레이블 로딩 시작 (차원 제한: 128-512)
2025-06-23 06:24:42,642 - INFO - 쿼리 실행 완료: 725725개 레코드 발견
   wavelet_vector 파싱: 100%|██████████| 725725/725725 [00:44<00:00, 16198.05it/s]
2025-06-23 06:25:27,447 - INFO - ✅ 'wavelet_vector' 로딩 완료: 725725개 벡터
2025-06-23 06:25:27,880 - INFO - 테이블 'dct_vector'에서 벡터와 레이블 로딩 시작 (차원 제한: 128-512)
2025-06-23 06:25:30,777 - INFO - 쿼리 실행 완료: 105560개 레코드 발견
   dct_vector 파싱: 100%|██████████| 105560/105560 [00:06<00:00, 15485.05it/s]
2025-06-23 06:25:37,596 - INFO - ✅ 'dct_vector' 로딩 완료: 105560개 벡터
2025-06-23 06:25:37,706 - INFO - 데이터 로딩 완료: 총 831285개 벡터, 최대 차원 511



📈 데이터 요약:
   🔸 총 벡터 수: 831,285개
   🔸 총 레이블 수: 831,285개
   🔸 고유 레이블 수: 5,736개
   🔸 최대 차원: 511


## 2단계: 데이터 전처리
이전과 동일하게 패딩 및 데이터 분할을 수행합니다.

In [4]:
# 벡터 패딩
logger.info(f"벡터 패딩 시작: 목표 차원 {max_dim}")
padded_vectors = [np.pad(vec, (0, max_dim - len(vec))) for vec in all_vectors]
X = torch.tensor(padded_vectors, dtype=torch.float32)
logger.info(f"벡터 패딩 및 텐서 변환 완료: {X.shape}")

# 훈련/검증 데이터 분할 (인덱스 기반)
n_samples = len(X)
indices = np.arange(n_samples)
train_indices, val_indices = train_test_split(
    indices, test_size=0.15, random_state=42, shuffle=True, stratify=all_labels
)
logger.info(f"데이터 분할 완료: 훈련 {len(train_indices):,}개, 검증 {len(val_indices):,}개")

# 메모리 정리
del padded_vectors
import gc
gc.collect()

2025-06-23 06:25:37,750 - INFO - 벡터 패딩 시작: 목표 차원 511
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13280\3328961750.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  X = torch.tensor(padded_vectors, dtype=torch.float32)
2025-06-23 06:26:29,545 - INFO - 벡터 패딩 및 텐서 변환 완료: torch.Size([831285, 511])
2025-06-23 06:26:30,176 - INFO - 데이터 분할 완료: 훈련 706,592개, 검증 124,693개


0

## 3단계: 개선된 Beta-VAE 모델 정의
복합 손실 함수와 KL Annealing을 지원하도록 모델 클래스를 수정합니다.

In [5]:
class ImprovedBetaVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=128, beta=1.0, dropout_rate=0.1, lambda_cosine=0.5):
        super(ImprovedBetaVAE, self).__init__()
        self.beta = beta
        self.lambda_cosine = lambda_cosine
        
        # 인코더
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        # 디코더
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, input_dim)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def loss_function(self, recon_x, x, mu, logvar, mask, kl_weight=1.0):
        # 1. 재구성 손실 (MSE)
        recon_loss_mse = F.mse_loss(recon_x * mask, x * mask, reduction='sum') / mask.sum()
        
        # 2. 재구성 손실 (Cosine Similarity)
        cos_sim = F.cosine_similarity(recon_x, x, dim=-1).mean()
        recon_loss_cos = self.lambda_cosine * (1 - cos_sim)
        
        # 총 재구성 손실
        recon_loss = recon_loss_mse + recon_loss_cos
        
        # 3. KL Divergence 손실 (KL Annealing 적용)
        kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        
        # 최종 손실
        total_loss = recon_loss + self.beta * kl_weight * kld
        
        return total_loss, recon_loss, kld, cos_sim

## 4단계: 훈련 준비 및 하이퍼파라미터 설정
개선된 전략에 맞춰 하이퍼파라미터를 설정합니다.

In [7]:
# 하이퍼파라미터 설정
input_dim = max_dim
latent_dim = 128
beta = 0.5            # KL 페널티를 낮춰 잠재 공간 활성화 유도
lambda_cosine = 0.5   # 코사인 유사도 손실 가중치
base_lr = 1e-4        # 안정적인 학습을 위해 학습률 하향 조정
weight_decay = 1e-5
epochs = 100
batch_size = 512
dropout_rate = 0.1
kl_warmup_epochs = 20 # KL Annealing을 위한 워밍업 에포크
patience = 15

# 모델, 옵티마이저, 스케줄러 등 초기화
vae = ImprovedBetaVAE(
    input_dim=input_dim, 
    latent_dim=latent_dim, 
    beta=beta,
    dropout_rate=dropout_rate,
    lambda_cosine=lambda_cosine
).to(device)

# Early Stopping 클래스 정의
class AdvancedEarlyStopping:
    def __init__(self, patience=7, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model)
        else:
            self.counter += 1
            
        if self.counter >= self.patience:
            if self.restore_best_weights:
                model.load_state_dict(self.best_weights)
            return True
        return False
    
    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

optimizer = optim.AdamW(vae.parameters(), lr=base_lr, weight_decay=weight_decay)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=epochs-kl_warmup_epochs, eta_min=base_lr*0.01)
early_stopping = AdvancedEarlyStopping(patience=patience)
scaler = GradScaler() if device.type == 'cuda' else None

# 마스크 및 데이터로더 생성
mask = torch.tensor([[1.0] * d + [0.0] * (max_dim - d) for d in original_dims], dtype=torch.float32)

train_dataset = data.TensorDataset(X[train_indices], mask[train_indices])
train_dataloader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4)

val_dataset = data.TensorDataset(X[val_indices], mask[val_indices])
val_dataloader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4)

logger.info("개선된 모델 및 훈련 설정 완료")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13280\385663320.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if device.type == 'cuda' else None
2025-06-23 06:27:31,257 - INFO - 개선된 모델 및 훈련 설정 완료


## 5단계: 개선된 훈련 루프
KL Annealing과 코사인 유사도 모니터링을 포함한 훈련 루프를 실행합니다.

In [8]:
# 훈련 기록용 리스트
history = {
    'train_loss': [], 'val_loss': [], 'recon_loss': [], 'kld_loss': [], 'cos_sim': [], 'lr': []
}

logger.info("개선된 모델 훈련 시작...")

for epoch in range(epochs):
    vae.train()
    
    # KL Annealing 가중치 계산
    kl_weight = min(1.0, epoch / kl_warmup_epochs)
    
    epoch_losses = {'total': 0, 'recon': 0, 'kld': 0, 'cos_sim': 0}
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for x_batch, m_batch in progress_bar:
        x_batch = x_batch.to(device, non_blocking=True)
        m_batch = m_batch.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        with autocast(enabled=scaler is not None):
            recon_batch, mu, logvar = vae(x_batch)
            total_loss, recon_loss, kld_loss, cos_sim = vae.loss_function(
                recon_batch, x_batch, mu, logvar, m_batch, kl_weight
            )
        
        if scaler:
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            total_loss.backward()
            optimizer.step()
        
        epoch_losses['total'] += total_loss.item()
        epoch_losses['recon'] += recon_loss.item()
        epoch_losses['kld'] += kld_loss.item()
        epoch_losses['cos_sim'] += cos_sim.item()
        
        progress_bar.set_postfix({
            'Loss': f"{total_loss.item():.4f}",
            'CosSim': f"{cos_sim.item():.4f}",
            'KLD': f"{kld_loss.item():.4f}",
            'KL_w': f"{kl_weight:.2f}"
        })
    
    # 에포크 평균 손실 기록
    history['train_loss'].append(epoch_losses['total'] / len(train_dataloader))
    history['recon_loss'].append(epoch_losses['recon'] / len(train_dataloader))
    history['kld_loss'].append(epoch_losses['kld'] / len(train_dataloader))
    
    # 검증 단계
    vae.eval()
    val_epoch_losses = {'total': 0, 'cos_sim': 0}
    with torch.no_grad():
        for x_batch, m_batch in val_dataloader:
            x_batch = x_batch.to(device, non_blocking=True)
            m_batch = m_batch.to(device, non_blocking=True)
            recon_batch, mu, logvar = vae(x_batch)
            total_loss, _, _, cos_sim = vae.loss_function(recon_batch, x_batch, mu, logvar, m_batch, kl_weight)
            val_epoch_losses['total'] += total_loss.item()
            val_epoch_losses['cos_sim'] += cos_sim.item()

    avg_val_loss = val_epoch_losses['total'] / len(val_dataloader)
    avg_val_cos_sim = val_epoch_losses['cos_sim'] / len(val_dataloader)
    history['val_loss'].append(avg_val_loss)
    history['cos_sim'].append(avg_val_cos_sim)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    print(f"Epoch {epoch+1} Summary: Val Loss: {avg_val_loss:.4f}, Val Cosine Sim: {avg_val_cos_sim:.4f}")
    
    # 스케줄러 및 조기 종료
    if epoch >= kl_warmup_epochs:
        cosine_scheduler.step()
        
    if early_stopping(avg_val_loss, vae, epoch):
        logger.info(f"Early stopping at epoch {epoch+1}")
        break

logger.info("훈련 완료.")

2025-06-23 06:27:31,282 - INFO - 개선된 모델 훈련 시작...
Epoch 1/100 [Train]:   0%|          | 0/1381 [00:00<?, ?it/s]C:\Users\Administrator\AppData\Local\Temp\ipykernel_13280\4254108318.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler is not None):
Epoch 1/100 [Train]: 100%|██████████| 1381/1381 [00:16<00:00, 82.37it/s, Loss=1.1950, CosSim=0.3490, KLD=417.8455, KL_w=0.00] 


Epoch 1 Summary: Val Loss: 1.0798, Val Cosine Sim: 0.4358


TypeError: __call__() takes 3 positional arguments but 4 were given

## 6단계: 잠재 공간 시각화 (UMAP)
훈련된 모델로 검증 데이터의 임베딩을 생성하고, UMAP을 사용해 2차원으로 시각화하여 클래스별 군집 형성 정도를 확인합니다.

In [ ]:
logger.info("잠재 공간 시각화를 위한 임베딩 추출 시작")
vae.eval()
val_embeddings = []
with torch.no_grad():
    for x_batch, _ in val_dataloader:
        x_batch = x_batch.to(device)
        mu, _ = vae.encode(x_batch)
        val_embeddings.append(mu.cpu().numpy())

val_embeddings = np.concatenate(val_embeddings, axis=0)
val_labels = np.array(all_labels)[val_indices]

# 시각화를 위해 데이터 일부 샘플링 (10000개)
sample_indices = np.random.choice(len(val_embeddings), size=10000, replace=False)
sampled_embeddings = val_embeddings[sample_indices]
sampled_labels = val_labels[sample_indices]

logger.info("UMAP으로 차원 축소 및 시각화 진행")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(sampled_embeddings)

# 시각화
df = pd.DataFrame({'x': embedding_2d[:, 0], 'y': embedding_2d[:, 1], 'label': sampled_labels})
unique_labels = df['label'].unique()
colors = cm.rainbow(np.linspace(0, 1, len(unique_labels)))
label_color_map = {label: color for label, color in zip(unique_labels, colors)}

plt.figure(figsize=(14, 10))
for label, group in df.groupby('label'):
    if len(group) > 5: # 너무 작은 그룹은 제외
        plt.scatter(group.x, group.y, c=[label_color_map[label]], label=label, s=10, alpha=0.7)

plt.title('UMAP Projection of the Latent Space', fontsize=18)
plt.xlabel('UMAP Dimension 1', fontsize=12)
plt.ylabel('UMAP Dimension 2', fontsize=12)
plt.grid(True)
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=2)
plt.show()

## 7단계: Recall@K 성능 측정 (faiss)
임베딩의 실제 검색 성능을 평가합니다. Faiss를 사용하여 ANN 검색을 시뮬레이션하고, Brute-force 검색 결과(Ground Truth)와 비교하여 Recall@K를 계산합니다.

In [ ]:
logger.info("Recall@K 측정을 위한 faiss 실험 시작")

# 검증 데이터셋을 데이터베이스와 쿼리로 분할
db_size = int(len(val_embeddings) * 0.9)
db_vectors = val_embeddings[:db_size]
query_vectors = val_embeddings[db_size:]
db_labels = val_labels[:db_size]
query_labels = val_labels[db_size:]

# 1. Ground Truth 생성 (Brute-force Exact Search)
d = db_vectors.shape[1]
index_exact = faiss.IndexFlatIP(d) # IP: Inner Product (Cosine Similarity)
faiss.normalize_L2(db_vectors) # 코사인 유사도 검색을 위해 정규화
index_exact.add(db_vectors)
faiss.normalize_L2(query_vectors)
D_exact, I_exact = index_exact.search(query_vectors, 1) # k=1 로 가장 가까운 이웃 탐색
ground_truth_labels = db_labels[I_exact.flatten()]

# 2. ANN 인덱스(IVFFlat) 생성 및 검색
nlist = 100  # IVF 클러스터 수
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(db_vectors)
index_ivf.add(db_vectors)

recall_results = {}
K_values = [1, 5, 10, 50, 100]

# nprobe 값에 따른 성능 변화 측정
for nprobe in [1, 5, 10, 20]:
    index_ivf.nprobe = nprobe
    D_ann, I_ann = index_ivf.search(query_vectors, max(K_values))
    
    recalls = []
    for k in K_values:
        correct_count = 0
        for i in range(len(query_vectors)):
            # 정답 레이블과 동일한 레이블이 Top-K 결과에 있는지 확인
            retrieved_labels = db_labels[I_ann[i, :k]]
            if query_labels[i] in retrieved_labels:
                correct_count += 1
        recalls.append(correct_count / len(query_vectors))
    recall_results[f'nprobe={nprobe}'] = recalls

# 결과 출력
recall_df = pd.DataFrame(recall_results, index=[f'Recall@{k}' for k in K_values])
print("\n--- Recall@K Performance (IVFFlat) ---")
print(recall_df)

recall_df.T.plot(kind='bar', figsize=(12, 7))
plt.title('Recall@K for different nprobe values')
plt.ylabel('Recall')
plt.xlabel('nprobe setting')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--')
plt.legend(title='K value')
plt.show()